In [ ]:
# Colab setup: install packages not preinstalled on Colab (safe to re-run)
!pip install -q abess

# Reproduce-then-Extend — State Policy Liberalism (American State Politics)  ·  **Day 1 tutorial**

> **Published study.** Caughey, D. & Warshaw, C. (2016). "The Dynamics of State Policy Liberalism, 1936–2014." *American Journal of Political Science* 60(4):899–913. The paper measures state **policy liberalism** and links it to public opinion and state political conditions — a cross-sectional OLS. This example illustrates **regularization and variable selection** in a small-sample (n = 50) design.

## Background

A central question in American state politics is why some states adopt systematically more **liberal policies**. The state-representation literature argues policy tracks **public opinion** (the opinion–policy link), mediated by **party control** and organized constituencies. We take the **cross-section of the 50 states** and ask which characteristics predict policy liberalism.

## Data and codebook

**Unit of analysis:** a U.S. **state** (cross-section averaged over 2005–2012); n = 50. Data from the **Correlates of State Policy Project** (Jordan & Grossmann 2020).

| Variable | Definition |
|---|---|
| `pollib_median` | state policy liberalism (Caughey–Warshaw), higher = more liberal **(outcome)** |
| `mood` | public policy mood (higher = more liberal) |
| `citi6013` | citizen ideology (Berry et al.), higher = more liberal |
| `ranney4_control` | Ranney index of Democratic party control |
| `union_density` | percent unionized |
| `evangelical_pop` | evangelical share of population |
| `gini_coef` | income inequality (Gini) |
| `gsppcap` | gross state product per capita |

## Descriptive results

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.linear_model import LassoCV, RidgeCV, ElasticNetCV, LinearRegression, lasso_path
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
rng = np.random.RandomState(2026)

In [ ]:
d = pd.read_csv('https://raw.githubusercontent.com/desmarais-lab/desmarais-lab.github.io/master/istanbul_bilgi_ml_files/data/state_policy_liberalism.csv')
outcome = 'pollib_median'
preds = [c for c in d.columns if c not in ('st', outcome)]
print(f'{d.shape[0]} states x {len(preds)} predictors (p/n = {len(preds)/d.shape[0]:.2f})')
d[[outcome]+preds].describe().T.round(2)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9, 3.2))
ax[0].hist(d[outcome], bins=12, color='#a6cee3', edgecolor='white'); ax[0].set_title('Outcome: policy liberalism')
cors = d[preds].corrwith(d[outcome]).sort_values()
ax[1].barh(cors.index, cors.values, color=['#1f78b4' if v>0 else '#e31a1c' for v in cors.values])
ax[1].axvline(0, color='grey'); ax[1].set_title('Correlation with outcome'); plt.tight_layout()

## Reproduce the published regression

The state-representation model regresses policy liberalism on public opinion and state political conditions. We expect the **opinion–policy link**: liberal mood/ideology, Democratic control, and union strength predict more liberal policy.

In [ ]:
ols = sm.OLS(d[outcome], sm.add_constant(d[preds])).fit()
print(ols.summary())
print(f'\nIn-sample R^2 = {ols.rsquared:.2f}')

## Regularization & variable selection

With only **50 states** and several predictors, ordinary least squares has few observations per coefficient and **over-fits**. Penalized regression — lasso ($L_1$), ridge ($L_2$), elastic net — shrinks the coefficients to trade a little bias for much lower variance and better prediction on new states.

In [ ]:
# Standardize predictors (sklearn does not standardize internally; glmnet does)
X = d[preds].values.astype(float); y = d[outcome].values.astype(float)
Xs = StandardScaler().fit_transform(X)
liCV = LassoCV(cv=5, random_state=0, max_iter=100000).fit(Xs, y)
n_keep = int(np.sum(liCV.coef_ != 0))
print(f'At the CV-optimal penalty the lasso keeps {n_keep} of {len(preds)} predictors (rest set to exactly 0).')
coefs = pd.DataFrame({'OLS(std)': LinearRegression().fit(Xs,y).coef_,
                      'Lasso': liCV.coef_,
                      'Ridge': RidgeCV(alphas=np.logspace(-2,3,40)).fit(Xs,y).coef_,
                      'ElasticNet': ElasticNetCV(cv=5, l1_ratio=0.5, random_state=0, max_iter=100000).fit(Xs,y).coef_},
                     index=preds).round(3)
coefs

**Which fits best out of sample?** With n = 50 a single split is noisy, so we **repeat a 70/30 split 50 times**: each replicate tunes the penalty by cross-validation on the training states and scores once on the held-out states.

In [ ]:
def rmse(a,b): return float(np.sqrt(mean_squared_error(a,b)))
REPS = 50; res = {k:[] for k in ['OLS','Lasso','Ridge','ElasticNet']}
for r in range(REPS):
    Xtr,Xte,ytr,yte = train_test_split(X, y, test_size=0.30, random_state=2025+r)
    sc = StandardScaler().fit(Xtr); Xtrs, Xtes = sc.transform(Xtr), sc.transform(Xte)
    res['OLS'].append(rmse(yte, LinearRegression().fit(Xtr,ytr).predict(Xte)))
    res['Lasso'].append(rmse(yte, LassoCV(cv=5,random_state=0,max_iter=100000).fit(Xtrs,ytr).predict(Xtes)))
    res['Ridge'].append(rmse(yte, RidgeCV(alphas=np.logspace(-2,3,40)).fit(Xtrs,ytr).predict(Xtes)))
    res['ElasticNet'].append(rmse(yte, ElasticNetCV(cv=5,l1_ratio=0.5,random_state=0,max_iter=100000).fit(Xtrs,ytr).predict(Xtes)))
mean = {k:np.mean(v) for k,v in res.items()}
for k,v in mean.items(): print(f'{k:12s} held-out RMSE = %.3f' % v)
best = min(['Lasso','Ridge','ElasticNet'], key=lambda k: mean[k])
print(f'\nBest penalization: {best}. {100*(mean["OLS"]-mean[best])/mean["OLS"]:.0f}% lower held-out RMSE than OLS.')

Because there are so few states per predictor, unpenalized OLS over-fits and the penalized models **predict new states more accurately**, while the lasso simultaneously **selects** the handful of state characteristics that carry the signal.

In [ ]:
# Lasso coefficient paths (standardized)
alphas, cpath, _ = lasso_path(Xs, y, n_alphas=40)
plt.figure(figsize=(6,3.4)); plt.plot(np.log10(alphas), cpath.T, color='#1f78b4', alpha=.4)
plt.xlabel('log10(alpha)  (more penalty ->)'); plt.ylabel('coefficient'); plt.title('Lasso paths'); plt.tight_layout()

## Best-subset selection with ABESS

Lasso reaches a sparse model through shrinkage. **Best-subset selection** searches directly for the subset of predictors that fits best; the **adaptive best-subset (ABESS)** algorithm does so efficiently and picks the subset size automatically.

In [ ]:
try:
    from abess.linear import LinearRegression as AbessLR
    ab = AbessLR().fit(Xs, y)
    sel = [p for p,c in zip(preds, ab.coef_) if c!=0]
    print(f'ABESS selects {len(sel)} of {len(preds)} predictors: {sel}')
except Exception as e:
    print('Install abess to run this cell:  !pip install abess')
    print('(', e, ')')

## Takeaway

This illustrates **regularization in a small-sample design**: with 50 states and several predictors, ordinary regression over-fits, and lasso/elastic-net plus best-subset selection predict new states more accurately **and** name the compact set of characteristics that drive the outcome — in contrast to the large-n civil-war/turnout models where regularization only ties.

## Recommended exercises

1. Add more Correlates-of-State-Policy predictors and watch the held-out gain over OLS grow with the predictor-to-state ratio.
2. Compare the CV-optimal (`lambda.min`-style) and sparser (`1-SE`) penalties: how much accuracy does the sparser model give up?
3. Refit predicting a different state outcome and report which predictors the lasso keeps.